In [27]:
import polars as pl
from utils import getConnection
from utils import cleanContracts
from utils import readCSV

con, dataset_path = getConnection() # Create the duckDB connection
readCSV(con, dataset_path) # Read CSV and create temp contracts table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# What does the data look like before cleaning?

In [28]:
con.sql("""
DESCRIBE contracts
""")

┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ reference_number        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ procurement_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vendor_name             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vendor_postal_code      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ buyer_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ contract_date           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ economic_object_code    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description_en          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description_fr

In [29]:
# Call encapsulated data cleaning function
cleanContracts(con) 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# What does the data look like after cleaning?

In [30]:
con.sql("""
DESCRIBE contracts_clean
""")

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ reference_number      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ economic_object_code  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vendor_name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ buyer_name            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ procurement_id        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ contract_date         │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ contract_period_start │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ delivery_date         │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ contract_value        │ DOUBLE      │ 

# One row per contract, representing the latest state of each contract

In [ ]:
# 'WHERE rn = 1' ensures we only get the most recent version of each contract
# Because contracts are grouped by reference number and ordered by date 
# in descending order. Total rows now equals 499k
con.sql("""
SELECT * FROM contracts_clean
WHERE rn = 1 
""")

┌───────────────────────┬──────────────────────┬───────────────────────────────────────────┬────────────────────┬─────────────────────┬───────────────┬───────────────────────┬───────────────┬────────────────┬────────────────┬─────────────────┬─────────────────────┬───────────────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┐
│   reference_number    │ economic_object_code │                vendor_name                │     buyer_name     │   procurement_id    │ contract_date │ contract_period_start │ delivery_date │ contract_value │ original_value │ amendment_value │ indigenous_business │ former_public_servant │  owner_org   │                                              owner_org_title                                               │  rn   │
│        varchar        │       varchar        │                  varchar                  │      varchar       │       varchar       │     date      │     

# Our data is now extremely workable, and ready for analysis